In [ ]:
"""
Student Name: Full Name
Student ID: 1234567
Assignment: 2
Date: YYYY-MM-DD
"""

In [ ]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score
)

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBM")

# TODO: use the last two digits of your student ID. Use 1 if they are 00.
STUDENT_SEED = 0
np.random.seed(STUDENT_SEED)

# Assignment 2: Regression and Classification

Course: COMP8831 Machine Learning  
Weighting: 10 percent of the final grade  
Total marks: 50  
Due: 25 September 2026, 11:59 PM

## Overview

You will use one dataset in two ways. First, predict a student's final percentage grade as a number. Then predict the same grade as a letter band.

Use the supplied model settings and your `STUDENT_SEED`. The notebook must run from top to bottom without errors. Keep all code, answers, tables and plots in the notebook.

## Load the dataset and create the two targets

The UCI dataset stores the final grade, `G3`, on a 0 to 20 scale. The supplied code converts it to a percentage. Do not change the target construction. An internet connection is required.

| Percentage | Grade band |
|---:|:---|
| 90 to 100 | A+ |
| 85 to below 90 | A |
| 80 to below 85 | A- |
| 75 to below 80 | B+ |
| 70 to below 75 | B |
| 65 to below 70 | B- |
| 60 to below 65 | C+ |
| 55 to below 60 | C |
| 50 to below 55 | C- |
| Below 50 | D |

Source: Cortez, P. (2008), Student Performance, UCI Machine Learning Repository, DOI 10.24432/C5TG7T.

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"

with urlopen(DATA_URL, timeout=30) as response:
    zip_bytes = response.read()

with ZipFile(BytesIO(zip_bytes)) as archive:
    with archive.open("student-por.csv") as csv_file:
        student = pd.read_csv(csv_file, sep=";")

GRADE_ORDER = ["D", "C-", "C", "C+", "B-", "B", "B+", "A-", "A", "A+"]
GRADE_BINS = [-np.inf, 50, 55, 60, 65, 70, 75, 80, 85, 90, np.inf]

student["grade_percent"] = student["G3"] * 5
student["grade_band"] = pd.cut(
    student["grade_percent"],
    bins=GRADE_BINS,
    labels=GRADE_ORDER,
    right=False,
    ordered=True,
)

student.head()

# Part 1: Exploratory data analysis and preparation (10 marks)

## Dataset inspection (2 marks)

1. Show the dataset shape, data types, first five rows and total missing values. (1)
2. Answer these questions in the markdown cell below. (1)
   - How many samples are in the dataset?
   - How many columns are in the original loaded table?
   - After removing the target-related columns, how many modelling features remain?
   - How many modelling features are numeric and how many are categorical?

In [ ]:
# TODO: inspect the dataset and answer the questions above.


**Dataset inspection answer:**  
Write your answer here.

## Explore the targets (3 marks)

1. Plot the distribution of `grade_percent`. Report its mean, median and standard deviation. (1)
2. Print all grade-band counts, draw a labelled count plot in `GRADE_ORDER`, and name the most common band. If you need help with plotting, see **Optional plotting help** at the end of this notebook. (1)
3. Plot at least two useful relationships between input features and `grade_percent`. A feature-to-grade relationship means placing one input feature, such as `G1`, `G2`, `studytime` or `absences`, on one axis and `grade_percent` on the other axis to see how the final grade changes. A scatter or regression plot suits a numeric feature; a box plot suits a categorical or small discrete feature. Write one brief observation for each plot. Optional examples are provided at the end. (1)

In [ ]:
# TODO: explore grade_percent with statistics and a distribution plot.


In [ ]:
# TODO: print grade-band counts and draw a count plot in GRADE_ORDER.


In [ ]:
# TODO: plot at least two input-feature-to-grade relationships.
# See the optional plotting examples at the end if you need a starting point.


**Target exploration observations:**  
Write your answer here.

## Problem type and target leakage (2 marks)

1. Which column is the regression target, and which column is the classification target? Then explain why predicting the first is regression and predicting the second is classification. Answer using this assignment, not only a general definition. (1)
2. Why must `G3`, `grade_percent` and `grade_band` all be removed from the input features? (1)

**Problem type and leakage answer:**  
Write your answer here.

## Split and preprocessing (3 marks)

1. Create `X`, `y_regression` and `y_classification`. Remove all three target-related columns from `X`. (Included with the leakage mark above.)
2. Use one 80/20 split for `X` and both targets, with `random_state=STUDENT_SEED` and stratification by `y_classification`. (1)
3. Identify the numeric and categorical feature columns. (1)
4. Run the supplied preprocessing cell below. It standardises numeric features and one-hot encodes categorical features. Use `preprocessor` inside each model pipeline. You do not need to write any functions. (1)

In [ ]:
# TODO: create X and both targets.
# X must not contain G3, grade_percent or grade_band.

# TODO: identify numeric_features and categorical_features.

# TODO: make one train-test split for X and both targets.


In [ ]:
# Supplied preprocessing. Run this cell before fitting the models.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ]
)
preprocessor


# Part 2: Regression (12 marks)

Predict `grade_percent` with these settings:

- `LinearRegression()`
- `RandomForestRegressor(n_estimators=200, random_state=STUDENT_SEED, n_jobs=1)`
- `LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=15, random_state=STUDENT_SEED, verbosity=-1, n_jobs=1)`

Fit and predict with all three models. Put `preprocessor` as the first step of each model pipeline. You may complete and run one model cell at a time; no custom functions are required. Report test MSE, RMSE, MAE and R2 for each model. (6 marks)

In [ ]:
regression_results = []
regression_predictions = {}

# TODO: fit and evaluate Linear Regression.


In [ ]:
# TODO: fit and evaluate Random Forest Regression. Keep the fitted model as rf_regression_model.


In [ ]:
# TODO: fit and evaluate LightGBM Regression.


## Regression questions (6 marks)

1. Which regression model performed best on your test set? Use at least two test metrics and compare it with another model. (2)
2. What does your Random Forest MSE mean for a student's percentage grade? Remember that MSE uses squared units. Use the RMSE to explain the error in percentage points. (2)
3. What does the Random Forest R2 mean for variation in students' final percentage grades? Explain why it is not the percentage of correct predictions. (2)

**Regression answers:**  
Write your answers here and include your values.

# Part 3: Classification (12 marks)

Predict `grade_band` with these settings:

- `LogisticRegression(max_iter=3000, class_weight="balanced", random_state=STUDENT_SEED)`
- `RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=STUDENT_SEED, n_jobs=1)`
- `LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15, class_weight="balanced", random_state=STUDENT_SEED, verbosity=-1, n_jobs=1)`

For each of the three models, put `preprocessor` as the first pipeline step, report test accuracy and the classification report, and draw a confusion matrix with `GRADE_ORDER`. You may complete and run one model cell at a time; no custom functions are required. (9 marks)

In [ ]:
classification_results = []
classification_predictions = {}

# TODO: fit and evaluate Logistic Regression.


In [ ]:
# TODO: fit and evaluate Random Forest Classification. Keep the fitted model as rf_classification_model.


In [ ]:
# TODO: fit and evaluate LightGBM Classification.


## Classification comparison (3 marks)

1. Which classifier performed best? Use its accuracy and either macro F1 or weighted F1. Do not use accuracy alone. (2)
2. Did accuracy and your chosen F1 measure support the same conclusion? Refer to your values. (1)

**Classification comparison answer:**  
Write your answer here.

# Part 4: Random Forest feature importance (5 marks)

Use the fitted Random Forest classifier.

1. Get the transformed feature names and match them to `feature_importances_`. (2)
2. Print all importances in descending order and plot the top ten. (1)
3. Name the three strongest transformed features and include their values. Explain what the forest used them for. State what feature importance does not tell us. (2)

In [ ]:
# TODO: print the sorted Random Forest classification importances and plot the top ten.


**Feature-importance answer:**  
Write your answer here.

# Part 5: Interpret the Random Forest classifier (6 marks)

Choose one grade band from your Random Forest classification report and confusion matrix. Use the actual values and counts from your output.

1. What does precision mean for your chosen band? (1)
2. What does recall mean for that band? (1)
3. What do its F1 score and support tell you? (1)
4. In your confusion matrix, what do the rows, columns and diagonal show? Describe one off-diagonal error using its count and both grade bands. (1)
5. What is a false positive for your chosen band in this problem? (1)
6. What is a false negative for that band, and what could be one practical consequence? (1)

Generic metric definitions receive no marks. Every answer must refer to this grade-prediction problem, a named band and your output.

**Random Forest interpretation:**  
Write your answers here.

# Part 6: Summary and reflection (5 marks)

1. Show a regression table with all three models and MSE, RMSE, MAE and R2. (1)
2. Show a classification table with all three models and accuracy, macro F1 and weighted F1. (1)
3. Name the best model for each target and support both choices with test values. (1)
4. Did regression or classification give the more useful view of student performance? Which would be more practical in a real school? There is no right or wrong choice; the mark is for a result-based, problem-specific justification. (1)
5. State one specific limitation or safeguard for using these results with real students. (1)

Generic answers receive no marks.

In [ ]:
# TODO: display your two final results tables.


**Summary and reflection:**  
Write your answers here.

# Submission

Submit one Jupyter notebook named `StudentID_YourName_Assignment_2.ipynb` through the Assignment 2 submission link on Moodle. Include your name and student ID, run every cell, and keep all outputs and plots inline.

## Use of generative AI

You may use generative AI to help generate or debug code. There is no mark deduction for using AI for code generation. If you use it, acknowledge the tool and identify the code cells or code sections it helped with. You are responsible for checking that the code runs and is correct.

Do not use generative AI to write the descriptive, explanation, interpretation, comparison or reflection answers. Those answers must be written in your own words and must refer to your own results.

**AI code acknowledgement, if used:** Tool used: ________. Code cells or sections: ________.

## Late submission

Ten percent is deducted within 24 hours of the deadline, and 20 percent after 24 and up to 48 hours. Work submitted more than 48 hours late is not marked unless Affected Performance Consideration applies. The APC form is at https://www.unitec.ac.nz/current-students/study-support/affected-performance-consideration

# Optional plotting help

Use these examples only if you need help with the Part 1 plots. You may copy an example into the relevant Part 1 code cell and change the feature name. These examples do not replace your written observations.

- The first example plots the grade bands in `GRADE_ORDER`.
- The second example shows a numeric feature-to-grade relationship. Each point is one student.
- The third example shows how grades are distributed at each value of a small discrete feature.

In [ ]:
# Optional example 1: grade-band counts and count plot
band_counts = student["grade_band"].value_counts(sort=False).reindex(GRADE_ORDER)
display(band_counts.rename("students").to_frame())

plt.figure(figsize=(9, 4))
ax = sns.countplot(data=student, x="grade_band", order=GRADE_ORDER)
ax.bar_label(ax.containers[0])
plt.xlabel("Grade band")
plt.ylabel("Number of students")
plt.title("Students in each grade band")
plt.show()

# Optional example 2: numeric feature-to-grade relationship
plt.figure(figsize=(7, 4))
sns.regplot(data=student, x="G2", y="grade_percent", scatter_kws={"alpha": 0.45})
plt.title("Second-period grade and final percentage grade")
plt.show()

# Optional example 3: small discrete feature-to-grade relationship
plt.figure(figsize=(7, 4))
sns.boxplot(data=student, x="studytime", y="grade_percent")
plt.title("Study-time category and final percentage grade")
plt.show()
